# Naive CUDA Matrix Multiplication — Compilation & Profiling

This notebook walks through:
1. Writing a **naive GEMM CUDA kernel** (one thread per output element)
2. Compiling it with **`nvcc`** targeting the A100 PCIe 40 GB (`sm_80`)
3. Running a correctness check against NumPy
4. Profiling with **NVIDIA Nsight Compute (`ncu`)**
5. Parsing and displaying the key profiling metrics

**Hardware target:** A100 PCIe 40 GB (compute capability 8.0)

**Prerequisites**
```
CUDA Toolkit ≥ 11.x  (nvcc, ncu)
Python packages  : numpy, subprocess (stdlib), pandas, matplotlib
```

## 0 — Environment Check

In [ ]:
import subprocess, sys, shutil

def run(cmd, **kw):
    """Run a shell command and print stdout/stderr."""
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True, **kw)
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr, file=sys.stderr)
    return result

# Verify tools exist
for tool in ('nvcc', 'ncu', 'nvidia-smi'):
    path = shutil.which(tool)
    status = f'✅  {path}' if path else '❌  NOT FOUND'
    print(f'{tool:15s} {status}')

print()
run('nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv,noheader')
run('nvcc --version')

## 1 — Write the CUDA Source

The kernel assigns **one thread per output element** `C[row, col]`.
Each thread accumulates the dot product of row `row` of `A` with column `col` of `B`.
This is intentionally naive — no shared memory, no tiling — which makes it a good
baseline to profile and then optimise.

In [ ]:
CUDA_SRC = r"""
#include <cuda_runtime.h>
#include <cstdio>
#include <cstdlib>
#include <cmath>

// ---------------------------------------------------------------------------
// Naive GEMM: C = A * B
//   A : M x K   (row-major)
//   B : K x N   (row-major)
//   C : M x N   (row-major)
// Each thread computes one element of C.
// ---------------------------------------------------------------------------
__global__ void matmul_naive(
        const float* __restrict__ A,
        const float* __restrict__ B,
              float* __restrict__ C,
        int M, int N, int K)
{
    int row = blockIdx.y * blockDim.y + threadIdx.y;  // output row
    int col = blockIdx.x * blockDim.x + threadIdx.x;  // output col

    if (row >= M || col >= N) return;

    float acc = 0.0f;
    for (int k = 0; k < K; ++k)
        acc += A[row * K + k] * B[k * N + col];

    C[row * N + col] = acc;
}

// ---------------------------------------------------------------------------
// Small host driver — called from Python via subprocess
// Usage: ./matmul_naive <M> <N> <K> <block_dim>
// ---------------------------------------------------------------------------
int main(int argc, char* argv[])
{
    int M  = (argc > 1) ? atoi(argv[1]) : 1024;
    int N  = (argc > 2) ? atoi(argv[2]) : 1024;
    int K  = (argc > 3) ? atoi(argv[3]) : 1024;
    int BD = (argc > 4) ? atoi(argv[4]) : 16;    // threads per block (each dim)

    printf("MatMul  M=%d  N=%d  K=%d  block=%dx%d\n", M, N, K, BD, BD);

    // ---- allocate & initialise host arrays --------------------------------
    size_t sA = (size_t)M * K * sizeof(float);
    size_t sB = (size_t)K * N * sizeof(float);
    size_t sC = (size_t)M * N * sizeof(float);

    float *hA = (float*)malloc(sA);
    float *hB = (float*)malloc(sB);
    float *hC = (float*)malloc(sC);

    for (int i = 0; i < M*K; ++i) hA[i] = (float)(i % 7) * 0.1f;
    for (int i = 0; i < K*N; ++i) hB[i] = (float)(i % 5) * 0.1f;

    // ---- allocate device arrays -------------------------------------------
    float *dA, *dB, *dC;
    cudaMalloc(&dA, sA);
    cudaMalloc(&dB, sB);
    cudaMalloc(&dC, sC);

    cudaMemcpy(dA, hA, sA, cudaMemcpyHostToDevice);
    cudaMemcpy(dB, hB, sB, cudaMemcpyHostToDevice);

    // ---- launch kernel ----------------------------------------------------
    dim3 block(BD, BD);
    dim3 grid((N + BD - 1) / BD, (M + BD - 1) / BD);

    // warm-up
    matmul_naive<<<grid, block>>>(dA, dB, dC, M, N, K);
    cudaDeviceSynchronize();

    // timed run
    cudaEvent_t t0, t1;
    cudaEventCreate(&t0); cudaEventCreate(&t1);
    cudaEventRecord(t0);

    matmul_naive<<<grid, block>>>(dA, dB, dC, M, N, K);

    cudaEventRecord(t1);
    cudaEventSynchronize(t1);

    float ms = 0;
    cudaEventElapsedTime(&ms, t0, t1);

    double flops  = 2.0 * M * N * K;
    double tflops = flops / (ms * 1e-3) / 1e12;
    printf("Kernel time : %.3f ms\n", ms);
    printf("Throughput  : %.4f TFLOP/s\n", tflops);

    // ---- copy result back -------------------------------------------------
    cudaMemcpy(hC, dC, sC, cudaMemcpyDeviceToHost);

    // spot-check C[0][0]
    float expected = 0;
    for (int k = 0; k < K; ++k) expected += hA[k] * hB[k * N];
    printf("C[0][0]     : %.6f  (expected %.6f)\n", hC[0], expected);

    // ---- cleanup ----------------------------------------------------------
    cudaFree(dA); cudaFree(dB); cudaFree(dC);
    free(hA); free(hB); free(hC);
    return 0;
}
"""

SRC_PATH = '/tmp/matmul_naive.cu'
with open(SRC_PATH, 'w') as f:
    f.write(CUDA_SRC)

print(f'Source written to {SRC_PATH}')
print(f'Lines: {len(CUDA_SRC.splitlines())}')

## 2 — Compile with `nvcc`

Key flags explained:

| Flag | Meaning |
|------|---------|
| `-arch=sm_80` | Ampere / A100 native ISA |
| `-O3` | Aggressive host-side optimisation |
| `--use_fast_math` | Fused multiply-add, fast reciprocals |
| `-lineinfo` | Embed source-line info for the profiler |
| `-Xptxas -v` | Print register/shared-memory usage per kernel |

In [ ]:
BIN_PATH = '/tmp/matmul_naive'

nvcc_cmd = (
    f'nvcc {SRC_PATH} '
    f'-o {BIN_PATH} '
    f'-arch=sm_80 '           # A100 PCIe
    f'-O3 '
    f'--use_fast_math '
    f'-lineinfo '             # source correlation for ncu
    f'-Xptxas -v '            # verbose PTX assembler stats
    f'-Xcompiler -Wall'       # host-side warnings
)

print('Running:', nvcc_cmd)
print('─' * 60)
result = run(nvcc_cmd)

if result.returncode == 0:
    print('\n✅  Compilation succeeded')
else:
    raise RuntimeError('nvcc compilation failed — see errors above')

## 3 — Correctness Check (CPU vs GPU)

Run the binary on a small matrix and compare `C[0][0]` reported by the GPU with
the value NumPy computes on the same data.

In [ ]:
import numpy as np

M, N, K = 64, 64, 64

# Run the GPU binary
result = run(f'{BIN_PATH} {M} {N} {K} 16')

# Replicate the same data as the C code
hA = np.array([(i % 7) * 0.1 for i in range(M * K)], dtype=np.float32).reshape(M, K)
hB = np.array([(i % 5) * 0.1 for i in range(K * N)], dtype=np.float32).reshape(K, N)
hC_ref = hA @ hB

print(f'NumPy  C[0][0] = {hC_ref[0, 0]:.6f}')
print('(compare with the "expected" value printed by the kernel above)')

## 4 — Timing Sweep: Matrix Size vs Throughput

Run the binary across several square-matrix sizes and plot throughput.

In [ ]:
import re
import matplotlib.pyplot as plt
import pandas as pd

sizes      = [256, 512, 1024, 2048, 4096]
block_dim  = 16          # 16×16 = 256 threads/block

records = []
for sz in sizes:
    r = run(f'{BIN_PATH} {sz} {sz} {sz} {block_dim}')
    out = r.stdout
    ms_match  = re.search(r'Kernel time\s*:\s*([\d.]+)', out)
    tf_match  = re.search(r'Throughput\s*:\s*([\d.]+)', out)
    ms  = float(ms_match.group(1))  if ms_match  else None
    tfl = float(tf_match.group(1))  if tf_match  else None
    records.append({'M=N=K': sz, 'time_ms': ms, 'TFLOP/s': tfl})
    print(f'  {sz:4d}³  →  {ms:8.2f} ms  {tfl:.4f} TFLOP/s')

df = pd.DataFrame(records)
print()
print(df.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(df['M=N=K'], df['time_ms'], 'o-', color='steelblue')
axes[0].set_title('Kernel Time vs Matrix Size')
axes[0].set_xlabel('M = N = K')
axes[0].set_ylabel('Time (ms)')
axes[0].grid(True, alpha=0.3)

# A100 PCIe 40 GB peak FP32 throughput ≈ 19.5 TFLOP/s
A100_PEAK_FP32 = 19.5
axes[1].plot(df['M=N=K'], df['TFLOP/s'], 's-', color='darkorange', label='Naive kernel')
axes[1].axhline(A100_PEAK_FP32, color='red', linestyle='--', alpha=0.6, label=f'A100 peak FP32 ({A100_PEAK_FP32} TFLOP/s)')
axes[1].set_title('FP32 Throughput vs Matrix Size')
axes[1].set_xlabel('M = N = K')
axes[1].set_ylabel('TFLOP/s')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/tmp/matmul_timing.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved to /tmp/matmul_timing.png')

## 5 — Profile with `ncu` (Nsight Compute)

We collect three metric **sections** in a single pass:

| Section | What it tells you |
|---------|-------------------|
| `SpeedOfLight` | Achieved % of peak memory bandwidth and compute |
| `MemoryWorkloadAnalysis` | L1/L2/DRAM traffic, cache hit-rates |
| `ComputeWorkloadAnalysis` | SM utilisation, warp efficiency |

> **Note:** `ncu` requires either root privileges or `CAP_SYS_ADMIN`.
> On most clusters you add `--target-processes all` and run as a privileged user.
> If your environment restricts this, `ncu` will print a clear error message.

In [ ]:
NCU_OUTPUT   = '/tmp/matmul_naive.ncu-rep'   # binary report (optional)
NCU_CSV      = '/tmp/matmul_ncu.csv'          # parsed metrics

PROFILE_M = PROFILE_N = PROFILE_K = 1024
BLOCK_DIM = 16

ncu_cmd = (
    f'ncu '
    f'--target-processes all '
    f'--kernel-name matmul_naive '
    f'--launch-count 1 '         # profile only the first kernel invocation
    f'--section SpeedOfLight '
    f'--section MemoryWorkloadAnalysis '
    f'--section ComputeWorkloadAnalysis '
    f'--csv '
    f'--log-file {NCU_CSV} '
    f'--export {NCU_OUTPUT} '
    f'{BIN_PATH} {PROFILE_M} {PROFILE_N} {PROFILE_K} {BLOCK_DIM}'
)

print('Running ncu — this may take 30–90 s ...')
print('Command:', ncu_cmd)
print('─' * 60)
result = run(ncu_cmd)

if result.returncode == 0:
    print('\n✅  ncu profiling succeeded')
else:
    print('\n⚠️   ncu returned non-zero — see output above.')
    print('Common causes: insufficient privileges, kernel not found, old driver.')

## 6 — Parse and Display ncu Results

In [ ]:
import os, io

if not os.path.exists(NCU_CSV):
    print('No CSV found — skipping parse step (did ncu succeed above?)')
else:
    # ncu CSV has comment lines starting with '"'
    with open(NCU_CSV) as f:
        lines = f.readlines()

    # Find the header row (starts with '"ID"')
    header_idx = next(i for i, l in enumerate(lines) if l.startswith('"ID"'))
    csv_body   = ''.join(lines[header_idx:])
    ncu_df     = pd.read_csv(io.StringIO(csv_body))

    print(f'Loaded {len(ncu_df)} metric rows')
    print('Columns:', list(ncu_df.columns))
    ncu_df.head()

In [ ]:
if 'ncu_df' in dir():
    # Key metrics to highlight — ncu column names vary slightly by version
    KEY_METRICS = [
        'sm__throughput.avg.pct_of_peak_sustained_elapsed',   # SM utilisation
        'l1tex__throughput.avg.pct_of_peak_sustained_elapsed',
        'lts__throughput.avg.pct_of_peak_sustained_elapsed',
        'gpu__dram_throughput.avg.pct_of_peak_sustained_elapsed',
        'smsp__warps_eligible.avg.pct_of_peak_sustained_active',  # warp occupancy
        'smsp__sass_thread_inst_executed_op_ffma_pred_on.sum',     # FMAs executed
    ]

    metric_col = 'Metric Name'   # adjust if your ncu version uses a different name
    value_col  = 'Metric Value'

    if metric_col in ncu_df.columns:
        subset = ncu_df[ncu_df[metric_col].isin(KEY_METRICS)][[metric_col, value_col, 'Unit']]
        if subset.empty:
            print('Key metrics not found — printing full table:')
            print(ncu_df[[metric_col, value_col]].to_string(index=False))
        else:
            print(subset.to_string(index=False))
    else:
        print(ncu_df.to_string())

## 7 — Interpreting the Results

The naive kernel has predictable bottlenecks that `ncu` will confirm:

| Observation | Root cause |
|-------------|------------|
| **Low SM utilisation** (typically < 5 % of 19.5 TFLOP/s peak) | Each thread re-fetches the same column of B repeatedly → no data reuse |
| **Low arithmetic intensity** | ~2·M·N·K FLOPs vs 2·(MK+KN)·4 B of DRAM reads → ≈ 0.25 FLOP/byte for square matrices |
| **Low L1/L2 hit rate** | Column-major access pattern for B strides across cache lines |
| **High DRAM bandwidth pressure** | Every element of A and B is re-read K and M times respectively |

### Next steps to optimise
1. **Shared-memory tiling** — load tiles of A and B into `__shared__` memory so each value is read from DRAM once per tile, reused `TILE_SIZE` times.
2. **Register blocking** — have each thread compute a small sub-tile of C, amortising the fetch cost further.
3. **Warp-level primitives / tensor cores** — use `wmma` API to leverage the A100's 312 TFLOP/s BF16 tensor cores.
4. **cuBLAS** — the vendor BLAS is the practical ceiling; use it as your performance reference.

## 8 — (Optional) Re-run ncu in Human-Readable Mode

If you prefer the colourised text report instead of CSV:

In [ ]:
ncu_text_cmd = (
    f'ncu '
    f'--target-processes all '
    f'--kernel-name matmul_naive '
    f'--launch-count 1 '
    f'--section SpeedOfLight '
    f'--section MemoryWorkloadAnalysis '
    f'--section ComputeWorkloadAnalysis '
    f'{BIN_PATH} {PROFILE_M} {PROFILE_N} {PROFILE_K} {BLOCK_DIM}'
)

print('Human-readable ncu output:')
print('─' * 60)
run(ncu_text_cmd)

## 9 — (Optional) Open the Report in Nsight Compute GUI

If you have a desktop environment, the binary report can be opened interactively:

```bash
ncu-ui /tmp/matmul_naive.ncu-rep
```

The GUI provides:
- **Roofline chart** showing where your kernel sits relative to memory and compute bounds
- **Source / PTX / SASS correlation** (thanks to `-lineinfo`)
- **Guided analysis** with actionable suggestions

In [ ]:
# Summary
import os
files = {
    'CUDA source'     : SRC_PATH,
    'Compiled binary' : BIN_PATH,
    'ncu CSV report'  : NCU_CSV,
    'ncu binary report': NCU_OUTPUT,
    'Timing plot'     : '/tmp/matmul_timing.png',
}
print('Generated artefacts:')
for label, path in files.items():
    exists = '✅' if os.path.exists(path) else '❌ (not found)'
    size   = f'{os.path.getsize(path):,} B' if os.path.exists(path) else ''
    print(f'  {exists}  {label:25s}  {path}  {size}')